<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_31_http_requests/note_lesson_31_http.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Урок 31 — HTTP: requests, httpx, aiohttp

| Крок | Що робимо |
|---|---|
| 0 | запускаємо навчальний API диспетчерської |
| 1 | шлях запиту: URL, DNS, HTTP як текст |
| 2 | `requests`: GET і параметри (вправа 1) |
| 3 | статус-коди: 404 (вправа 2) |
| 4 | POST, JSON і токен (вправа 3) |
| 5 | тайм-аут (вправа 4) |
| 6 | повторні спроби з паузою (вправа 5) |
| 7 | одночасні запити: `httpx.AsyncClient` + `gather` (вправа 6) |
| 8 | свій клієнт до API (вправа 7) |
| 9 | справжній API: PyPI (вправа 8) |
| 10 | знайди помилку (вправа 9) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія й схеми — у книзі: [Урок 31](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_31/).

---
## 0. Навчальний API

Сервер диспетчерської — файл `smachno_api.py` поруч із цим ноутбуком. Він написаний на стандартній бібліотеці і працює у фоновому потоці прямо в ядрі ноутбука, без інтернету.

У Colab ноутбук відкривається **без** файлів репозиторію — тоді клітинка нижче завантажить `smachno_api.py` з GitHub. Бібліотеки `requests` і `httpx` у Colab уже є, `aiohttp` клітинка встановить.

| Запит | Що робить |
|---|---|
| `GET /restaurants` | список ресторанів; `?district=Поділ` — фільтр |
| `GET /restaurants/<id>` | один ресторан або 404 |
| `GET /restaurants/<id>/status?delay=0.5` | «чи відкрито?» із затримкою |
| `POST /orders` | створити замовлення; заголовок `Authorization: Bearer smachno-token` |
| `GET /flaky` | перші 2 запити — 503, далі — 200 |
| `GET /slow` | відповідь через 3 секунди |
| `POST /reset` | скинути замовлення й лічильник `/flaky` |

In [ ]:
import importlib.util
import os
import subprocess
import sys
import urllib.request

if not os.path.exists("smachno_api.py"):
    url = ("https://raw.githubusercontent.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/"
           "main/module_4/lessons/lesson_31_http_requests/smachno_api.py")
    urllib.request.urlretrieve(url, "smachno_api.py")

for package in ["requests", "httpx", "aiohttp"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
print("готово")

In [ ]:
import asyncio
import socket
import time

import aiohttp
import httpx
import requests

from smachno_api import start_server

BASE = start_server()
TOKEN = "smachno-token"          # навчальний; справжній токен — лише зі змінних середовища
print(BASE)

---
## 1. Шлях запиту: URL → DNS → TCP → HTTP

`urlsplit` розбирає адресу на частини, `socket.getaddrinfo` робить DNS-запит: ім'я → IP-адреси.

In [ ]:
from urllib.parse import urlsplit

url = urlsplit("https://pypi.org/pypi/requests/json?format=short")
print(url.scheme, url.hostname, url.port, url.path, url.query)

try:
    print(sorted({info[4][0] for info in socket.getaddrinfo("pypi.org", 443, type=socket.SOCK_STREAM)}))
except socket.gaierror as error:
    print("DNS не відповів (немає інтернету?):", error)
print(socket.gethostbyname("localhost"))

**🔮 Прогноз:** `url.port` надрукував `None`. Яким портом піде запит і чому?

<details>
<summary>Відповідь</summary>

443: порт не вказано в URL, а для схеми `https` стандартний порт — 443 (для `http` — 80).

</details>

HTTP — це текст. Надішлемо запит звичайним сокетом, без бібліотек:

In [ ]:
request = (
    "GET /restaurants/3 HTTP/1.1\r\n"
    "Host: 127.0.0.1:8031\r\n"
    "Connection: close\r\n"
    "\r\n"
)
with socket.create_connection(("127.0.0.1", 8031)) as sock:
    sock.sendall(request.encode())
    response = b""
    while chunk := sock.recv(1024):
        response += chunk
print(response.decode())

**🔮 Прогноз:** Що зміниться у першому рядку відповіді, якщо попросити `/restaurants/42`?

<details>
<summary>Відповідь</summary>

`HTTP/1.1 404 Not Found` замість `HTTP/1.1 200 OK`, а в тілі — `{"error": "ресторан 42 не знайдено"}`. Перевір — зміни шлях у клітинці вище.

</details>

---
## 2. requests: GET і параметри

`requests.get(url, params={...}, timeout=5)`; `response.status_code`, `response.json()`. Параметри в URL `requests` кодує сам.

### Вправа 1

Напиши `names_in(district)` — список назв ресторанів району в порядку, як повертає API. Один запит, фільтр — параметром `district`, обов'язково `timeout`.

In [ ]:
def names_in(district):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    response = requests.get(f"{BASE}/restaurants", params={"district": district}, timeout=5)
    response.raise_for_status()
    return [restaurant["name"] for restaurant in response.json()]
    # END SOLUTION


assert names_in("Оболонь") == ["Суші Оболонь"]
assert names_in("Центр") == ["Вареники 24/7", "Шаурма Центр"]
assert names_in("Троєщина") == []
print("✅ Вправа 1 пройдена")

---
## 3. Статус-коди

`requests` **не** викидає виняток на 4xx/5xx: відповідь просто має інший `status_code`. `response.raise_for_status()` перетворює 4xx/5xx на `requests.HTTPError`.

In [ ]:
response = requests.get(f"{BASE}/restaurants/42", timeout=5)
print(response.status_code, response.ok, response.json())

### Вправа 2

`restaurant_or_none(restaurant_id)` повертає словник ресторану, `None` для 404, а для інших помилок (5xx) — викидає `requests.HTTPError`.

In [ ]:
def restaurant_or_none(restaurant_id):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}", timeout=5)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()
    # END SOLUTION


assert restaurant_or_none(4) == {"id": 4, "name": "Вареники 24/7", "district": "Центр"}
assert restaurant_or_none(42) is None
print("✅ Вправа 2 пройдена")

---
## 4. POST, JSON і токен

`requests.post(url, json=дані, headers={"Authorization": f"Bearer {TOKEN}"}, timeout=5)`. `json=` сам серіалізує словник і додає `Content-Type: application/json`.

| Код | Що означає тут |
|---|---|
| `201` | замовлення створено, у тілі — `id` |
| `401` | немає або неправильний токен |
| `422` | дані не пройшли перевірку: `{"errors": [...]}` |

### Вправа 3

`create_order(customer, total, restaurant_id=1, token=TOKEN)` надсилає замовлення і повертає пару `(status_code, тіло як dict)`. Якщо `token` — `None`, заголовок `Authorization` не надсилається.

In [ ]:
def create_order(customer, total, restaurant_id=1, token=TOKEN):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    order = {"restaurant_id": restaurant_id, "customer": customer, "total": total}
    response = requests.post(f"{BASE}/orders", json=order, headers=headers, timeout=5)
    return response.status_code, response.json()
    # END SOLUTION


requests.post(f"{BASE}/reset", timeout=5)
status, body = create_order("Олена", 420, token=None)
assert status == 401
status, body = create_order("Олена", 420, token="wrong")
assert status == 401
status, body = create_order("", -1, restaurant_id=99)
assert status == 422 and len(body["errors"]) == 3, body
status, body = create_order("Олена", 420)
assert status == 201 and body["id"] == 101 and body["status"] == "new", body
print("✅ Вправа 3 пройдена")

**🔮 Прогноз:** Ти двічі поспіль викликав `create_order("Олена", 420)`. Скільки замовлень створиться і які в них `id`?

<details>
<summary>Відповідь</summary>

Два — 102 і 103 (після 101 з вправи). `POST` не ідемпотентний: кожен виклик створює нове замовлення. Тому `POST` не можна автоматично повторювати після тайм-ауту.

</details>

---
## 5. Тайм-аут

Без `timeout` `requests` чекає **вічно**. `requests.Timeout` — спільний предок `ConnectTimeout` і `ReadTimeout`.

### Вправа 4

`get_json_or_none(path, timeout)` робить GET на `BASE + path` і повертає JSON; якщо сервер не відповів за `timeout` секунд — повертає `None`.

In [ ]:
def get_json_or_none(path, timeout):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        response = requests.get(f"{BASE}{path}", timeout=timeout)
    except requests.Timeout:
        return None
    response.raise_for_status()
    return response.json()
    # END SOLUTION


start = time.perf_counter()
assert get_json_or_none("/slow", timeout=0.5) is None
assert time.perf_counter() - start < 2, "тайм-аут не спрацював"
assert get_json_or_none("/restaurants/2", timeout=5)["name"] == "Піца Поділ"
print("✅ Вправа 4 пройдена")

---
## 6. Повторні спроби з паузою

Повторюємо **лише** 5xx і збої мережі, **лише** для GET, обмежену кількість разів, подвоюючи паузу. 4xx повертаємо одразу — повтор нічого не змінить.

### Вправа 5

`get_with_retry(path, attempts=4, pause=0.05)` повертає пару `(response, скільки спроб зроблено)`. Відповідь `< 500` — повертай одразу. Якщо всі спроби дали 5xx — поверни останню відповідь.

In [ ]:
def get_with_retry(path, attempts=4, pause=0.05):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    for attempt in range(1, attempts + 1):
        response = requests.get(f"{BASE}{path}", timeout=2)
        if response.status_code < 500 or attempt == attempts:
            return response, attempt
        time.sleep(pause)
        pause *= 2
    # END SOLUTION


requests.post(f"{BASE}/reset", timeout=5)
response, tries = get_with_retry("/flaky")
assert response.status_code == 200 and tries == 3, (response.status_code, tries)

response, tries = get_with_retry("/restaurants/42")
assert response.status_code == 404 and tries == 1, "4xx не повторюємо"

requests.post(f"{BASE}/reset", timeout=5)
response, tries = get_with_retry("/flaky", attempts=2)
assert response.status_code == 503 and tries == 2
print("✅ Вправа 5 пройдена")

---
## 7. Одночасні запити

Спершу по черзі: 5 статусів по 0,5 секунди.

In [ ]:
start = time.perf_counter()
for restaurant_id in range(1, 6):
    requests.get(f"{BASE}/restaurants/{restaurant_id}/status", params={"delay": 0.5}, timeout=5)
print(f"по черзі: {time.perf_counter() - start:.1f} с")

У Jupyter цикл подій уже працює, тому корутину запускаємо просто `await корутина()` — без `asyncio.run` (урок 27).

### Вправа 6

`check_all(ids, delay)` — **одночасно** отримує статуси ресторанів через `httpx.AsyncClient` і `asyncio.gather` і повертає список словників у порядку `ids`.

In [ ]:
async def check_all(ids, delay=0.5):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    async with httpx.AsyncClient(base_url=BASE, timeout=5) as client:
        async def fetch(restaurant_id):
            response = await client.get(f"/restaurants/{restaurant_id}/status", params={"delay": delay})
            response.raise_for_status()
            return response.json()

        return await asyncio.gather(*(fetch(restaurant_id) for restaurant_id in ids))
    # END SOLUTION


start = time.perf_counter()
statuses = await check_all([5, 4, 3, 2, 1])
elapsed = time.perf_counter() - start
print(f"одночасно: {elapsed:.1f} с")
assert [status["id"] for status in statuses] == [5, 4, 3, 2, 1]
assert all(status["open"] for status in statuses)
assert elapsed < 1.5, "запити йшли по черзі?"
print("✅ Вправа 6 пройдена")

**🔮 Прогноз:** Скільки часу займуть 20 ресторанів з `delay=0.5` через `check_all`? А 20 по черзі?

<details>
<summary>Відповідь</summary>

Приблизно 0,5 секунди одночасно (усі чекають разом) проти 10 секунд по черзі. На справжньому чужому сервері стільки одночасних запитів треба обмежувати: `asyncio.Semaphore` або `httpx.Limits`.

</details>

Те саме з `aiohttp` — інший інтерфейс, той самий ефект:

In [ ]:
async def check_all_aiohttp(ids, delay=0.5):
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=5)) as session:
        async def fetch(restaurant_id):
            url = f"{BASE}/restaurants/{restaurant_id}/status"
            async with session.get(url, params={"delay": delay}) as response:
                response.raise_for_status()
                return await response.json()

        return await asyncio.gather(*(fetch(restaurant_id) for restaurant_id in ids))


start = time.perf_counter()
statuses = await check_all_aiohttp(range(1, 6))
print(f"aiohttp: {time.perf_counter() - start:.1f} с", [status["name"] for status in statuses])

---
## 8. Свій клієнт до API

Уся робота з API — в одному класі: адреса, токен, тайм-аут; HTTP-коди перетворюються на **свої** винятки, тож решта програми нічого не знає про `httpx`.

### Вправа 7

Допиши `_request`: після відповіді 404 → `NotFound(текст з поля "error")`, 422 → `ValidationFailed(список з поля "errors")`, інші 4xx/5xx → `SmachnoError`, інакше — повернути JSON.

In [ ]:
class SmachnoError(Exception):
    pass


class NotFound(SmachnoError):
    pass


class ValidationFailed(SmachnoError):
    def __init__(self, errors):
        super().__init__("; ".join(errors))
        self.errors = errors


class SmachnoClient:
    def __init__(self, base_url, token, timeout=5):
        self._http = httpx.Client(base_url=base_url, timeout=timeout,
                                  headers={"Authorization": f"Bearer {token}"})

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self._http.close()

    def _request(self, method, path, **kwargs):
        response = self._http.request(method, path, **kwargs)
        # YOUR CODE HERE
        # BEGIN SOLUTION
        if response.status_code == 404:
            raise NotFound(response.json()["error"])
        if response.status_code == 422:
            raise ValidationFailed(response.json()["errors"])
        if response.is_error:
            raise SmachnoError(f"{method} {path}: {response.status_code}")
        return response.json()
        # END SOLUTION

    def restaurant(self, restaurant_id):
        return self._request("GET", f"/restaurants/{restaurant_id}")

    def create_order(self, restaurant_id, customer, total):
        return self._request("POST", "/orders",
                             json={"restaurant_id": restaurant_id, "customer": customer, "total": total})


requests.post(f"{BASE}/reset", timeout=5)
with SmachnoClient(BASE, TOKEN) as api:
    assert api.restaurant(3)["name"] == "Суші Оболонь"
    try:
        api.restaurant(42)
        raise AssertionError("очікували NotFound")
    except NotFound as error:
        assert "42" in str(error)
    try:
        api.create_order(1, "", 100)
        raise AssertionError("очікували ValidationFailed")
    except ValidationFailed as error:
        assert error.errors == ["customer: обов'язкове поле"]
    assert api.create_order(2, "Марко", 300)["id"] == 101

with SmachnoClient(BASE, "wrong-token") as api:
    try:
        api.create_order(2, "Марко", 300)
        raise AssertionError("очікували SmachnoError")
    except NotFound:
        raise AssertionError("401 — це не NotFound")
    except SmachnoError as error:
        assert "401" in str(error)
print("✅ Вправа 7 пройдена")

**🔮 Прогноз:** Чому в клієнт з книги повтори додані лише для `GET`, а не для `create_order`?

<details>
<summary>Відповідь</summary>

Якщо `POST /orders` не дочекався відповіді, замовлення могло вже створитися. Автоматичний повтор створить дубль — клієнт отримає два замовлення.

</details>

---
## 9. Справжній API: PyPI

Потрібен інтернет. `https://pypi.org/pypi/<пакет>/json` → поле `info.version` — остання версія; неіснуючий пакет — 404.

### Вправа 8

`latest_versions(packages)` — **одночасно** питає PyPI і повертає `{назва: версія}`; для неіснуючого пакета — `None`. Інші помилки мають вилітати винятком.

In [ ]:
async def latest_versions(packages):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    async with httpx.AsyncClient(base_url="https://pypi.org", timeout=10) as client:
        async def fetch(name):
            response = await client.get(f"/pypi/{name}/json")
            if response.status_code == 404:
                return name, None
            response.raise_for_status()
            return name, response.json()["info"]["version"]

        return dict(await asyncio.gather(*(fetch(name) for name in packages)))
    # END SOLUTION


try:
    versions = await latest_versions(["requests", "httpx", "no-such-package-xyz-31"])
except httpx.TransportError as error:
    print("⚠️ Немає доступу до pypi.org — вправу перевір, коли з'явиться інтернет:", type(error).__name__)
else:
    print(versions)
    assert set(versions) == {"requests", "httpx", "no-such-package-xyz-31"}
    assert versions["no-such-package-xyz-31"] is None
    assert versions["requests"].count(".") >= 1 and versions["httpx"].count(".") >= 1
    print("✅ Вправа 8 пройдена")

---
## 10. Знайди помилку

```python
def restaurant_name(restaurant_id):
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}")
    return response.json()["name"]
```

Для 42 функція падає з `KeyError: 'name'`. Тут **дві** проблеми: одну видно, друга проявиться, коли сервер зависне.

### Вправа 9

Виправ: для неіснуючого ресторану має вилітати `requests.HTTPError` (а не `KeyError`), і функція не має чекати відповідь довше 5 секунд.

In [ ]:
def restaurant_name(restaurant_id):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    response = requests.get(f"{BASE}/restaurants/{restaurant_id}", timeout=5)
    response.raise_for_status()
    return response.json()["name"]
    # END SOLUTION


assert restaurant_name(2) == "Піца Поділ"
try:
    restaurant_name(42)
    raise AssertionError("очікували HTTPError")
except requests.HTTPError as error:
    assert error.response.status_code == 404
print("✅ Вправа 9 пройдена")

---
## Самоперевірка

1. Які кроки між `requests.get("https://pypi.org/...")` і словником у Python?
2. Що поверне `requests` на 404 — виняток чи відповідь?
3. Чому `timeout` обов'язковий?
4. Що можна повторювати, а що ні?
5. Чому 5 запитів з `gather` займають стільки ж, скільки один?

<details>
<summary>Відповіді</summary>

1. URL → DNS (ім'я → IP) → TCP-з'єднання на порт 443 → TLS → HTTP-запит і відповідь → `json()`.
2. Відповідь зі `status_code == 404`; виняток — лише після `raise_for_status()`.
3. Без нього `requests` чекатиме завислий сервер вічно.
4. 5xx, тайм-аути, збої мережі — для ідемпотентних запитів (GET). Не повторюємо 4xx і автоматично — `POST`.
5. Запити не рахують, а чекають мережу; поки чекає один, цикл подій відправляє інші.

</details>

## Далі

- Книга: [Урок 31](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_31/) — DNS, порти, TCP і TLS, `Session`, ієрархія винятків, архітектура клієнта, `requests` vs `httpx` vs `aiohttp`.
- Документація: [requests](https://requests.readthedocs.io/en/latest/user/quickstart/), [httpx](https://www.python-httpx.org/quickstart/), [aiohttp client](https://docs.aiohttp.org/en/stable/client_quickstart.html).
- **Урок 32** — REST: принципи дизайну API — той самий обмін, але з боку того, хто проєктує сервер.